<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Fuwsegformer-finetuning/notebooks/5_segformer_mit_b0_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline 3: SegFormer plano (MiT-B0)

**Task 4.7 · Goal 2.3**

Este notebook entrena y evalúa SegFormer MiT-B0 con el mismo entorno legacy, 
split, imágenes CLAHE, máscaras provisionales y presupuesto de optimización del 
fine-tuning de UWSegFormer. La única diferencia deliberada es el decoder estándar 
`SegFormerHead`, sin los módulos especializados de UWSegFormer.

> **Estado del artefacto:** notebook preparado y validado estructuralmente. Debe 
ejecutarse en Colab con GPU y el Drive del proyecto montado. Los resultados no se 
prellenan: se generan desde checkpoints y predicciones reales.


## 1. Contrato de comparabilidad

El experimento aborta si no encuentra el manifiesto creado por UWSegFormer. Así 
se garantiza que cada `sample_id` conserva exactamente el mismo split y ground truth.

| Factor | UWSegFormer | SegFormer plano |
|---|---|---|
| Split | `dataset_manifest.csv`, seed 42 | El mismo manifiesto |
| Entrada | CLAHE/fallback ya resuelto | Misma ruta original |
| Máscara | Ground truth provisional actual | Misma máscara |
| Resolución | 640×480 | 640×480 |
| Iteraciones | 2,000 | 2,000 |
| Optimizador | AdamW, 6e-5 | AdamW, 6e-5 |
| Encoder | Congelado | Congelado |
| Métricas | mIoU, mDice | Las mismas + foreground y borde |


## 2. Entorno CondaColab reproducible


In [ ]:
# Ejecutar una sola vez; condacolab reiniciará el runtime.
try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError('Este notebook está diseñado para Google Colab.') from exc
try:
    import condacolab
    print('condacolab ya está disponible.')
except ImportError:
    !pip install -q condacolab==0.1.9
    import condacolab
    condacolab.install()


In [ ]:
# Mismo stack que UWSegFormer.
import subprocess
envs_output = subprocess.check_output(['conda', 'env', 'list'], text=True)
existing = {line.split()[0] for line in envs_output.splitlines() if line and not line.startswith('#')}
if 'uws' not in existing:
    !conda create -n uws python=3.8 -y
    !conda run -n uws python -m pip install --upgrade pip
    !conda run -n uws python -m pip install torch==1.10.2+cu111 torchvision==0.11.3+cu111 -f https://download.pytorch.org/whl/torch_stable.html
    !conda run -n uws python -m pip install mmcv-full==1.5.3 -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.10/index.html
    !conda run -n uws python -m pip install numpy==1.23.5 opencv-python-headless==4.7.0.72 matplotlib pandas
else:
    print("Reutilizando el entorno 'uws'.")


In [ ]:
!conda run -n uws python -c "import sys, torch, mmcv; print('Python:', sys.version.split()[0]); print('Torch:', torch.__version__); print('MMCV:', mmcv.__version__); assert sys.version_info[:2] == (3, 8); assert torch.__version__.startswith('1.10.2'); assert mmcv.__version__ == '1.5.3'; assert torch.cuda.is_available(), 'Activa GPU en Runtime > Change runtime type'"


## 3. Parámetros, rutas y código base


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
MAX_ITERS = 2000
EVAL_INTERVAL = 100
CHECKPOINT_INTERVAL = 250
SAMPLES_PER_GPU = 2
WORKERS_PER_GPU = 2
LEARNING_RATE = 6e-5
FREEZE_BACKBONE = True

DRIVE_ROOT = Path('/content/drive/MyDrive/Under_Water_Dataset')
UWS_EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments' / 'uwsegformer_binary_head_finetune'
SOURCE_MANIFEST = UWS_EXPERIMENT_ROOT / 'dataset_manifest.csv'
EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments' / 'segformer_mit_b0_plain'
DATASET_ROOT = Path('/content/segformer_binary_dataset')
UWSEGFORMER_ROOT = Path('/content/UWSegFormer')
PRETRAINED_CHECKPOINT = Path('/content/segformer_mit_b0_ade20k.pth')
PRETRAINED_URL = 'https://download.openmmlab.com/mmsegmentation/v0.5/segformer/segformer_mit-b0_512x512_160k_ade20k/segformer_mit-b0_512x512_160k_ade20k_20210726_101530-8ffa8fda.pth'
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

if not SOURCE_MANIFEST.exists():
    raise FileNotFoundError(
        f'No existe {SOURCE_MANIFEST}. Ejecuta primero el notebook 4 de UWSegFormer; '
        'este baseline no debe inventar otro split.'
    )
manifest = pd.read_csv(SOURCE_MANIFEST)
required = {'sample_id', 'split', 'original_image', 'original_mask'}
if not required.issubset(manifest.columns):
    raise ValueError(f'El manifiesto no contiene {sorted(required - set(manifest.columns))}')
if set(manifest.split) != {'training', 'validation'}:
    raise ValueError(f'Splits inesperados: {sorted(manifest.split.unique())}')
manifest_bytes = SOURCE_MANIFEST.read_bytes()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
print('SHA-256 del split compartido:', manifest_sha256)
display(manifest.groupby(['split', 'source']).size().unstack(fill_value=0))


In [ ]:
# Código fijado al mismo commit usado por el notebook de UWSegFormer.
UWS_COMMIT = 'b61fa450bad58137b3ab8fecd1e7f7cedba4ce57'
if not UWSEGFORMER_ROOT.exists():
    !git clone https://github.com/SAWRJJ/UWSegFormer.git /content/UWSegFormer
!git -C /content/UWSegFormer checkout {UWS_COMMIT}

decode_init = UWSEGFORMER_ROOT / 'mmseg/models/decode_heads/__init__.py'
text = decode_init.read_text()
text = text.replace(
    'from .segformer_head import PPMobileSegHead_segformer2,PPMobileSegHead_segformer1',
    'from .segformer_head import MAA_segformer1',
)
text = text.replace(
    "    'PPMobileSegHead_segformer2',\n    'PPMobileSegHead_segformer1'",
    "    'MAA_segformer1'",
)
decode_init.write_text(text)
for config_path in (UWSEGFORMER_ROOT / 'local_configs/uwsegformer').glob('*.py'):
    config_path.write_text(config_path.read_text().replace('../../_base_/', '../_base_/'))

if not PRETRAINED_CHECKPOINT.exists():
    !wget -q --show-progress -O {PRETRAINED_CHECKPOINT} {PRETRAINED_URL}
# Verifica el registro antes de lanzar un entrenamiento largo. El nombre distingue mayúsculas.
!PYTHONPATH=/content/UWSegFormer conda run -n uws python -c "from mmseg.models.builder import HEADS; assert HEADS.get('SegFormerHead') is not None; print('OK: SegFormerHead registrado')
print('Checkpoint oficial SegFormer-B0:', PRETRAINED_CHECKPOINT)


## 4. Materializar exactamente el mismo dataset

Las rutas se leen del manifiesto de UWSegFormer. No se vuelve a emparejar, sortear 
ni asignar muestras. Las máscaras se binarizan con el mismo criterio `> 0`.


In [ ]:
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
for split in ('training', 'validation'):
    (DATASET_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / 'annotations' / split).mkdir(parents=True, exist_ok=True)

audit_rows = []
for row in manifest.itertuples(index=False):
    image = cv2.imread(str(row.original_image), cv2.IMREAD_COLOR)
    mask = cv2.imread(str(row.original_mask), cv2.IMREAD_UNCHANGED)
    if image is None or mask is None:
        raise FileNotFoundError(f'No se pudo leer imagen/máscara de {row.sample_id}')
    if mask.ndim == 3:
        mask = np.max(mask[..., :3], axis=2)
    mask = (mask > 0).astype(np.uint8)
    if mask.shape != image.shape[:2]:
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
    image_out = DATASET_ROOT / 'images' / row.split / f'{row.sample_id}.jpg'
    mask_out = DATASET_ROOT / 'annotations' / row.split / f'{row.sample_id}.png'
    if not cv2.imwrite(str(image_out), image, [cv2.IMWRITE_JPEG_QUALITY, 95]):
        raise IOError(image_out)
    if not cv2.imwrite(str(mask_out), mask):
        raise IOError(mask_out)
    audit_rows.append({'sample_id': row.sample_id, 'split': row.split, 'foreground_ratio': float(mask.mean())})

audit = pd.DataFrame(audit_rows)
assert len(audit) == len(manifest)
display(audit.groupby('split').foreground_ratio.describe())


In [ ]:
# Inspección visual antes de entrenar.
sample = manifest.groupby('split', group_keys=False).apply(
    lambda frame: frame.sample(min(3, len(frame)), random_state=SEED)
).reset_index(drop=True)
fig, axes = plt.subplots(len(sample), 3, figsize=(14, 3.6 * len(sample)))
if len(sample) == 1:
    axes = np.array([axes])
for ax_row, row in zip(axes, sample.itertuples(index=False)):
    image = cv2.cvtColor(cv2.imread(str(DATASET_ROOT / 'images' / row.split / f'{row.sample_id}.jpg')), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(DATASET_ROOT / 'annotations' / row.split / f'{row.sample_id}.png'), cv2.IMREAD_GRAYSCALE)
    overlay = image.copy()
    overlay[mask == 1] = (0.45 * overlay[mask == 1] + 0.55 * np.array([255, 70, 40])).astype(np.uint8)
    ax_row[0].imshow(image); ax_row[0].set_title(f'{row.sample_id} · {row.split}')
    ax_row[1].imshow(mask, cmap='gray', vmin=0, vmax=1); ax_row[1].set_title('Ground truth')
    ax_row[2].imshow(overlay); ax_row[2].set_title('Superposición')
    for ax in ax_row: ax.axis('off')
plt.tight_layout()
plt.show()


## 5. SegFormer MiT-B0 plano y entrenamiento


In [ ]:
settings = {
    'seed': SEED, 'max_iters': MAX_ITERS, 'eval_interval': EVAL_INTERVAL,
    'checkpoint_interval': CHECKPOINT_INTERVAL, 'samples_per_gpu': SAMPLES_PER_GPU,
    'workers_per_gpu': WORKERS_PER_GPU, 'learning_rate': LEARNING_RATE,
    'freeze_backbone': FREEZE_BACKBONE, 'dataset_root': str(DATASET_ROOT),
    'experiment_root': str(EXPERIMENT_ROOT), 'checkpoint': str(PRETRAINED_CHECKPOINT),
    'manifest': str(SOURCE_MANIFEST), 'manifest_sha256': manifest_sha256,
}
Path('/content/segformer_settings.json').write_text(json.dumps(settings, indent=2))
display(pd.Series(settings, name='value'))


In [ ]:
TRAIN_SCRIPT = Path('/content/train_segformer_mit_b0.py')
TRAIN_SCRIPT.write_text(r'''
import json, re, sys, time
from pathlib import Path
import numpy as np
np.float = float
np.int = int
import mmcv
import torch
from mmcv.runner import load_checkpoint
from mmcv.utils import Config, get_git_hash
ROOT = Path('/content/UWSegFormer')
sys.path.insert(0, str(ROOT))
from mmseg import __version__
from mmseg.apis import set_random_seed, train_segmentor
from mmseg.datasets import build_dataset
from mmseg.models import build_segmentor
from mmseg.utils import get_root_logger

s = json.loads(Path('/content/segformer_settings.json').read_text())
norm = dict(type='BN', requires_grad=True)
cfg = Config(dict(
    model=dict(
        type='EncoderDecoder',
        pretrained=None,
        backbone=dict(type='mit_b0', style='pytorch'),
        decode_head=dict(
            type='SegFormerHead', in_channels=[32, 64, 160, 256],
            in_index=[0, 1, 2, 3], feature_strides=[4, 8, 16, 32],
            channels=256, decoder_params=dict(embed_dim=256), dropout_ratio=0.1,
            num_classes=2, norm_cfg=norm, align_corners=False,
            loss_decode=dict(type='CrossEntropyLoss', use_sigmoid=False, loss_weight=1.0),
        ),
        train_cfg=dict(), test_cfg=dict(mode='whole'),
    )
))
img_norm = dict(mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375], to_rgb=True)
crop = (480, 640)
train_pipeline = [
    dict(type='LoadImageFromFile'), dict(type='LoadAnnotations', reduce_zero_label=False),
    dict(type='Resize', img_scale=(640, 480), ratio_range=(0.8, 1.2)),
    dict(type='RandomCrop', crop_size=crop, cat_max_ratio=0.90),
    dict(type='RandomFlip', prob=0.5), dict(type='Normalize', **img_norm),
    dict(type='Pad', size=crop, pad_val=0, seg_pad_val=255),
    dict(type='DefaultFormatBundle'), dict(type='Collect', keys=['img', 'gt_semantic_seg']),
]
test_pipeline = [dict(type='LoadImageFromFile'), dict(type='MultiScaleFlipAug', img_scale=(640, 480), flip=False, transforms=[
    dict(type='Resize', keep_ratio=True), dict(type='RandomFlip'), dict(type='Normalize', **img_norm),
    dict(type='Pad', size=crop, pad_val=0, seg_pad_val=255),
    dict(type='ImageToTensor', keys=['img']), dict(type='Collect', keys=['img']),
])]
common = dict(type='CustomDataset', data_root=s['dataset_root'], img_suffix='.jpg', seg_map_suffix='.png',
              classes=('background', 'structure'), palette=[[0, 0, 0], [255, 80, 40]])
cfg.data = dict(samples_per_gpu=s['samples_per_gpu'], workers_per_gpu=s['workers_per_gpu'],
    train=dict(**common, img_dir='images/training', ann_dir='annotations/training', pipeline=train_pipeline),
    val=dict(**common, img_dir='images/validation', ann_dir='annotations/validation', pipeline=test_pipeline),
    test=dict(**common, img_dir='images/validation', ann_dir='annotations/validation', pipeline=test_pipeline))
cfg.optimizer = dict(type='AdamW', lr=s['learning_rate'], betas=(0.9, 0.999), weight_decay=0.01)
cfg.optimizer_config = dict()
cfg.lr_config = dict(policy='poly', warmup='linear', warmup_iters=min(100, s['max_iters']//10), warmup_ratio=1e-3, power=1.0, min_lr=1e-7, by_epoch=False)
cfg.runner = dict(type='IterBasedRunner', max_iters=s['max_iters'])
cfg.checkpoint_config = dict(by_epoch=False, interval=s['checkpoint_interval'], max_keep_ckpts=3)
cfg.evaluation = dict(interval=s['eval_interval'], metric=['mIoU', 'mDice'], pre_eval=False, save_best='mIoU', rule='greater')
cfg.log_config = dict(interval=20, hooks=[dict(type='TextLoggerHook', by_epoch=False)])
cfg.workflow = [('train', 1)]
cfg.work_dir = s['experiment_root']; cfg.gpu_ids = [0]; cfg.seed = s['seed']
cfg.resume_from = None; cfg.load_from = None; cfg.log_level = 'INFO'; cfg.find_unused_parameters = True
mmcv.mkdir_or_exist(cfg.work_dir)
cfg.dump(str(Path(cfg.work_dir) / 'segformer_mit_b0_plain_config.py'))
timestamp = time.strftime('%Y%m%d_%H%M%S', time.localtime())
logger = get_root_logger(log_file=str(Path(cfg.work_dir) / f'{timestamp}.log'), log_level=cfg.log_level)
set_random_seed(cfg.seed, deterministic=True)
model = build_segmentor(cfg.model)

# El checkpoint oficial ADE20K usa el esquema moderno `backbone.layers.*`, mientras
# este fork legacy usa `patch_embed1`, `block1`, ..., `norm4`. Convertimos todo
# el encoder de forma explícita. El decoder legacy se inicializa desde cero.
checkpoint = torch.load(s['checkpoint'], map_location='cpu')
state = checkpoint.get('state_dict', checkpoint)
state = {k.replace('module.', '', 1) if k.startswith('module.') else k: v for k, v in state.items()}
removed = {k: tuple(v.shape) for k, v in state.items() if k in {
    'decode_head.linear_pred.weight', 'decode_head.linear_pred.bias',
    'decode_head.conv_seg.weight', 'decode_head.conv_seg.bias'}}

def convert_modern_mit_backbone(source):
    converted, consumed = {}, set()
    for key, value in source.items():
        patch = re.fullmatch(r'backbone\.layers\.(\d+)\.0\.(projection|norm)\.(weight|bias)', key)
        if patch:
            stage, module, parameter = patch.groups(); stage = int(stage) + 1
            legacy_module = 'proj' if module == 'projection' else 'norm'
            converted[f'backbone.patch_embed{stage}.{legacy_module}.{parameter}'] = value
            consumed.add(key); continue
        stage_norm = re.fullmatch(r'backbone\.layers\.(\d+)\.2\.(weight|bias)', key)
        if stage_norm:
            stage, parameter = stage_norm.groups(); stage = int(stage) + 1
            converted[f'backbone.norm{stage}.{parameter}'] = value
            consumed.add(key); continue
        block = re.fullmatch(r'backbone\.layers\.(\d+)\.1\.(\d+)\.(.+)', key)
        if not block:
            continue
        stage, index, suffix = block.groups(); prefix = f'backbone.block{int(stage)+1}.{index}'
        if suffix in {'attn.attn.in_proj_weight', 'attn.attn.in_proj_bias'}:
            parameter = 'weight' if suffix.endswith('weight') else 'bias'
            dim = value.shape[1] if parameter == 'weight' else value.numel() // 3
            converted[f'{prefix}.attn.q.{parameter}'] = value[:dim].clone()
            converted[f'{prefix}.attn.kv.{parameter}'] = value[dim:].clone()
        else:
            replacements = {
                'attn.attn.out_proj.': 'attn.proj.',
                'attn.sr.': 'attn.sr.',
                'attn.norm.': 'attn.norm.',
                'ffn.layers.0.': 'mlp.fc1.',
                'ffn.layers.1.': 'mlp.dwconv.dwconv.',
                'ffn.layers.4.': 'mlp.fc2.',
                'norm1.': 'norm1.', 'norm2.': 'norm2.',
            }
            target_suffix = None
            for modern, legacy in replacements.items():
                if suffix.startswith(modern):
                    target_suffix = legacy + suffix[len(modern):]
                    break
            if target_suffix is None:
                continue
            converted[f'{prefix}.{target_suffix}'] = value
        consumed.add(key)
    unconsumed = sorted(k for k in source if k.startswith('backbone.') and k not in consumed)
    return converted, unconsumed

converted, unconsumed = convert_modern_mit_backbone(state)
current = model.state_dict()
shape_mismatches = {k: (tuple(v.shape), tuple(current[k].shape)) for k, v in converted.items() if k in current and tuple(v.shape) != tuple(current[k].shape)}
matched = {k: v for k, v in converted.items() if k in current and tuple(v.shape) == tuple(current[k].shape)}
backbone_total = sum(v.numel() for k, v in current.items() if k.startswith('backbone.'))
backbone_matched = sum(v.numel() for k, v in matched.items() if k.startswith('backbone.'))
coverage = backbone_matched / max(backbone_total, 1)
missing_backbone = sorted(k for k in current if k.startswith('backbone.') and k not in matched)
print(f'Cobertura del encoder convertido: {coverage:.2%}')
print('Clasificador ADE20K ignorado:', removed)
print('Claves modernas no consumidas:', unconsumed[:20])
print('Incompatibilidades de forma:', shape_mismatches)
print('Claves legacy faltantes:', missing_backbone[:20])
if coverage < 0.99:
    raise RuntimeError(f'Cobertura insuficiente del encoder convertido: {coverage:.2%}')
incompatible = model.load_state_dict(matched, strict=False)
unexpected_backbone = [k for k in incompatible.unexpected_keys if k.startswith('backbone.')]
missing_after_load = [k for k in incompatible.missing_keys if k.startswith('backbone.')]
if unexpected_backbone or missing_after_load:
    raise RuntimeError(f'Carga incompleta. Missing={missing_after_load}; unexpected={unexpected_backbone}')
if s['freeze_backbone']:
    for parameter in model.backbone.parameters(): parameter.requires_grad = False
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros congelados: {frozen:,}; entrenables: {trainable:,}')
datasets = [build_dataset(cfg.data.train)]
cfg.checkpoint_config['meta'] = dict(mmseg_version=f'{__version__}+{get_git_hash()[:7]}', config=cfg.pretty_text,
    CLASSES=datasets[0].CLASSES, PALETTE=datasets[0].PALETTE, manifest_sha256=s['manifest_sha256'])
model.CLASSES = datasets[0].CLASSES; model.PALETTE = datasets[0].PALETTE
train_segmentor(model, datasets, cfg, distributed=False, validate=True, timestamp=timestamp,
                meta=dict(seed=cfg.seed, exp_name='segformer_mit_b0_plain', manifest_sha256=s['manifest_sha256']))
''')
print(TRAIN_SCRIPT)


In [ ]:
# Entrenamiento supervisado en T4.
!LD_LIBRARY_PATH=/usr/local/envs/uws/lib:$LD_LIBRARY_PATH conda run -n uws python /content/train_segformer_mit_b0.py


## 6. Evaluación cuantitativa y eficiencia


In [ ]:
# Curvas y mejor checkpoint según mIoU.
log_files = sorted(EXPERIMENT_ROOT.glob('*.log.json'), key=lambda p: p.stat().st_mtime)
if not log_files:
    raise FileNotFoundError(f'No se encontraron logs en {EXPERIMENT_ROOT}')
rows = []
for line in log_files[-1].read_text().splitlines():
    try: rows.append(json.loads(line))
    except json.JSONDecodeError: pass
history = pd.DataFrame(rows)
train_history = history[history.get('mode', pd.Series(index=history.index, dtype=str)).eq('train')]
val_history = history[history.get('mode', pd.Series(index=history.index, dtype=str)).eq('val')]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if {'iter', 'loss'}.issubset(train_history.columns):
    axes[0].plot(train_history.iter, train_history.loss, color='#D55E00')
axes[0].set(title='Pérdida de entrenamiento', xlabel='Iteración', ylabel='Cross-Entropy Loss')
for metric, color in [('mIoU', '#0072B2'), ('mDice', '#009E73')]:
    if metric in val_history:
        axes[1].plot(val_history.iter, val_history[metric], marker='o', label=metric, color=color)
axes[1].set(title='Validación sobre el split compartido', xlabel='Iteración', ylabel='Score', ylim=(0, 1))
axes[1].legend()
for ax in axes: ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig(EXPERIMENT_ROOT / 'training_curves.png', dpi=180); plt.show()
display(val_history[[c for c in ['iter', 'mIoU', 'mDice', 'mAcc', 'aAcc'] if c in val_history]].dropna(how='all'))


In [ ]:
# Inferencia por imagen y métricas foreground/borde/latencia.
EVAL_SCRIPT = Path('/content/evaluate_segformer_mit_b0.py')
EVAL_SCRIPT.write_text(r'''
import json, sys, time
from pathlib import Path
import cv2, numpy as np, torch
np.float = float; np.int = int
ROOT = Path('/content/UWSegFormer'); sys.path.insert(0, str(ROOT))
from mmcv import Config
from mmseg.apis import inference_segmentor, init_segmentor
s = json.loads(Path('/content/segformer_settings.json').read_text())
work = Path(s['experiment_root'])
cfg = Config.fromfile(str(work / 'segformer_mit_b0_plain_config.py'))
best = work / 'best_mIoU.pth'
if not best.exists():
    candidates = sorted(work.glob('iter_*.pth'), key=lambda p: int(p.stem.split('_')[-1]))
    if not candidates: raise FileNotFoundError('No hay checkpoint entrenado')
    best = candidates[-1]
model = init_segmentor(cfg, str(best), device='cuda:0')
pred_dir = work / 'validation_predictions'; pred_dir.mkdir(exist_ok=True)
tp=fp=fn=tn=0; btp=bfp=bfn=0; latencies=[]; per_image=[]
kernel = np.ones((5, 5), np.uint8)  # tolerancia de 2 px
images = sorted((Path(s['dataset_root'])/'images'/'validation').glob('*.jpg'))
for i, image_path in enumerate(images):
    if i == 0: inference_segmentor(model, str(image_path)); torch.cuda.synchronize()
    start=time.perf_counter(); pred=np.asarray(inference_segmentor(model, str(image_path))[0], dtype=np.uint8); torch.cuda.synchronize()
    latencies.append((time.perf_counter()-start)*1000)
    gt=cv2.imread(str(Path(s['dataset_root'])/'annotations'/'validation'/f'{image_path.stem}.png'), cv2.IMREAD_GRAYSCALE)
    pred=(pred>0).astype(np.uint8); gt=(gt>0).astype(np.uint8)
    cv2.imwrite(str(pred_dir/f'{image_path.stem}.png'), pred)
    itp=int(((pred==1)&(gt==1)).sum()); ifp=int(((pred==1)&(gt==0)).sum())
    ifn=int(((pred==0)&(gt==1)).sum()); itn=int(((pred==0)&(gt==0)).sum())
    tp+=itp; fp+=ifp; fn+=ifn; tn+=itn
    pred_b=pred ^ cv2.erode(pred, np.ones((3,3),np.uint8)); gt_b=gt ^ cv2.erode(gt, np.ones((3,3),np.uint8))
    pred_match=(pred_b>0)&(cv2.dilate(gt_b,kernel)>0); gt_match=(gt_b>0)&(cv2.dilate(pred_b,kernel)>0)
    ibtp=int(pred_match.sum()); ibfp=int((pred_b>0).sum())-ibtp; ibfn=int((gt_b>0).sum())-int(gt_match.sum())
    btp+=ibtp; bfp+=ibfp; bfn+=ibfn
    per_image.append({'sample_id':image_path.stem,'iou_structure':itp/max(itp+ifp+ifn,1),'dice_structure':2*itp/max(2*itp+ifp+ifn,1)})
precision=tp/max(tp+fp,1); recall=tp/max(tp+fn,1); boundary_precision=btp/max(btp+bfp,1); boundary_recall=btp/max(btp+bfn,1)
metrics={'checkpoint':str(best),'manifest_sha256':s['manifest_sha256'],'n_validation':len(images),
 'iou_structure':tp/max(tp+fp+fn,1),'dice_structure':2*tp/max(2*tp+fp+fn,1),
 'precision_structure':precision,'recall_structure':recall,'specificity_background':tn/max(tn+fp,1),
 'boundary_f1_2px':2*boundary_precision*boundary_recall/max(boundary_precision+boundary_recall,1e-12),
 'latency_ms_median':float(np.median(latencies)),'latency_ms_p95':float(np.percentile(latencies,95)),
 'parameters_m':sum(p.numel() for p in model.parameters())/1e6}
(work/'evaluation_metrics.json').write_text(json.dumps(metrics,indent=2))
import csv
with (work/'per_image_metrics.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=per_image[0].keys()); writer.writeheader(); writer.writerows(per_image)
print(json.dumps(metrics,indent=2))
''')
!LD_LIBRARY_PATH=/usr/local/envs/uws/lib:$LD_LIBRARY_PATH conda run -n uws python /content/evaluate_segformer_mit_b0.py
metrics = json.loads((EXPERIMENT_ROOT / 'evaluation_metrics.json').read_text())
display(pd.Series(metrics))


## 7. Comparación directa con UWSegFormer


In [ ]:
seg_metrics = json.loads((EXPERIMENT_ROOT / 'evaluation_metrics.json').read_text())
uws_metrics_path = UWS_EXPERIMENT_ROOT / 'evaluation_metrics.json'
if uws_metrics_path.exists():
    uws_metrics = json.loads(uws_metrics_path.read_text())
    if uws_metrics.get('manifest_sha256') not in (None, manifest_sha256):
        raise ValueError('UWSegFormer y SegFormer no usan el mismo manifiesto.')
    comparison = pd.DataFrame({'SegFormer MiT-B0': seg_metrics, 'UWSegFormer': uws_metrics})
    numeric = comparison.apply(pd.to_numeric, errors='coerce').dropna(how='all')
    numeric['delta_segformer_minus_uws'] = numeric['SegFormer MiT-B0'] - numeric['UWSegFormer']
    display(numeric)
    numeric.to_csv(EXPERIMENT_ROOT / 'comparison_vs_uwsegformer.csv')
else:
    print(f'Falta {uws_metrics_path}. El notebook 4 registra métricas agregadas; ejecuta allí el mismo evaluador por píxel para una comparación completa.')


## 8. Criterio de cierre

El baseline queda **entrenado y evaluado** solo cuando existen en Drive:

- `best_mIoU.pth` o un checkpoint final;
- `evaluation_metrics.json`;
- `per_image_metrics.csv`;
- `training_curves.png`;
- el SHA-256 del mismo manifiesto usado por UWSegFormer.

No se concluye superioridad por una sola métrica ni con cuatro imágenes de validación. 
Se reportan IoU/Dice de estructura, F1 de borde a 2 px, precisión, recall, especificidad, 
latencia y parámetros. Task 4.8 debe repetir este notebook con el ground truth final.
